AI-Based Proxy Attendance Detection System

In [ ]:
!pip install gradio

In [ ]:
import pandas as pd
import numpy as np
import gradio as gr
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

In [ ]:
def proxy_attendance_detection(file):

    # Step 1: Read CSV
    df = pd.read_csv(file.name)

    # Step 2: Column Fix
    df = df.rename(columns={
        "Device_ID": "RFID_Reader_ID",
        "Location": "Room_ID"
    })

    # Step 3: Required columns check
    required_columns = ["Student_ID", "Date", "Time", "Subject", "Room_ID", "RFID_Reader_ID"]

    missing_columns = [col for col in required_columns if col not in df.columns]

    if missing_columns:
        return pd.DataFrame({
            "Error": [f"Missing columns: {missing_columns}"]
        }), None

    # Step 4: Time cleaning
    df["Time"] = pd.to_datetime(df["Time"], format="%H:%M:%S", errors="coerce")

    df["Hour"] = df["Time"].dt.hour
    df["Minute"] = df["Time"].dt.minute
    df["Second"] = df["Time"].dt.second

    df["Time_In_Seconds"] = (
        df["Hour"] * 3600 +
        df["Minute"] * 60 +
        df["Second"]
    )

    # Step 5: Sort
    df = df.sort_values(["Date", "Subject", "Room_ID", "RFID_Reader_ID", "Time"])

    # Step 6: Time Gap
    df["Previous_Time"] = df.groupby(
        ["Date", "Subject", "Room_ID", "RFID_Reader_ID"]
    )["Time"].shift(1)

    df["Time_Gap_Seconds"] = (df["Time"] - df["Previous_Time"]).dt.total_seconds()
    df["Time_Gap_Seconds"] = df["Time_Gap_Seconds"].fillna(999)

    # Step 7: Features
    df["Tight_Punch"] = (df["Time_Gap_Seconds"] <= 3).astype(int)

    df["Reader_Punch_Count"] = df.groupby(
        ["Date", "Subject", "Room_ID", "RFID_Reader_ID"]
    )["Student_ID"].transform("count")

    df["Student_Punch_Count"] = df.groupby(
        ["Date", "Subject", "Room_ID", "Student_ID"]
    )["Student_ID"].transform("count")

    df["Duplicate_Punch"] = (df["Student_Punch_Count"] > 1).astype(int)

    # Step 8: ML Model
    from sklearn.preprocessing import StandardScaler
    from sklearn.ensemble import IsolationForest

    features = [
        "Time_In_Seconds",
        "Time_Gap_Seconds",
        "Tight_Punch",
        "Reader_Punch_Count",
        "Duplicate_Punch"
    ]

    X = df[features].fillna(0)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model = IsolationForest(
        n_estimators=100,
        contamination=0.05,
        random_state=42
    )

    df["Prediction"] = model.fit_predict(X_scaled)

    df["Flag"] = df["Prediction"].map({
        1: "Normal",
        -1: "Suspicious Proxy"
    })

    # Step 9: Risk Score
    scores = model.decision_function(X_scaled)

    df["Risk_Score"] = (
        (scores.max() - scores) /
        (scores.max() - scores.min()) * 100
    ).round(2)

    # Step 10: Reason
    def generate_reason(row):
        reasons = []

        if row["Tight_Punch"] == 1:
            reasons.append("Very fast card punching")

        if row["Duplicate_Punch"] == 1:
            reasons.append("Same student punched multiple times")

        if row["Reader_Punch_Count"] > 70:
            reasons.append("High RFID reader load")

        if len(reasons) == 0:
            return "Normal"

        return ", ".join(reasons)

    df["Reason"] = df.apply(generate_reason, axis=1)

    # Step 11: Output
    suspicious_df = df[df["Flag"] == "Suspicious Proxy"][
        ["Student_ID", "Date", "Time", "Subject", "Room_ID", "RFID_Reader_ID", "Risk_Score", "Reason"]
    ]

    suspicious_df.to_csv("output.csv", index=False)

    return suspicious_df, "output.csv"

Anomaly Detection      Abnormal data detect करना
Unsupervised Learning  बिना label data से सीखना  


In [ ]:
app = gr.Interface(
    fn=proxy_attendance_detection,
    inputs=gr.File(label="Upload RFID Attendance CSV File"),
    outputs=[
        gr.Dataframe(label="Suspicious Students Shortlist"),
        gr.File(label="Download Report CSV")
    ],
    title="AI-Based Proxy Attendance Detection System",
    description="Upload attendance CSV file and get suspicious proxy list."
)

app.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://456c5d6129b2aab444.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
app.launch(debug=True)